# Probability Calibration of ML Classifiers for SMS Spam Detection
Eksperimen untuk jurnal: *Evaluating and Improving Probability Calibration of Machine Learning Classifiers for SMS Spam Detection*

**Alur:** load data -> split -> 4 model x 3 varian (Raw, Platt, Isotonic) -> metrik (Accuracy, F1, Brier, Log Loss, ECE) -> reliability diagram -> tabel siap tempel ke IEEE.

Jalankan sel dari atas ke bawah. Runtime: kira-kira 3-8 menit di Colab (CPU biasa cukup).

In [ ]:
import io, os, zipfile, urllib.request, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import accuracy_score, f1_score, brier_score_loss, log_loss, roc_auc_score

warnings.filterwarnings("ignore")
os.makedirs("outputs", exist_ok=True)

## 1. Load dataset (SMS Spam Collection, UCI)
Kalau download gagal, upload file `SMSSpamCollection` secara manual (lihat sel di bawah).

In [ ]:
URL = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"

def load_sms():
    try:
        raw = urllib.request.urlopen(URL, timeout=60).read()
        with zipfile.ZipFile(io.BytesIO(raw)) as z:
            with z.open("SMSSpamCollection") as f:
                return pd.read_csv(f, sep="\t", header=None, names=["label", "text"])
    except Exception as e:
        print("Download gagal:", e)
        print("Upload file 'SMSSpamCollection' lewat panel Files di Colab, lalu jalankan ulang sel ini.")
        return pd.read_csv("SMSSpamCollection", sep="\t", header=None, names=["label", "text"])

df = load_sms()
df["y"] = (df["label"] == "spam").astype(int)
print(df.shape)
print(df["label"].value_counts())
print(df["label"].value_counts(normalize=True).round(3))
df.head()

## 2. Model, kalibrasi, dan metrik
- **Raw**: model dilatih biasa, probabilitas apa adanya.
  - `LinearSVC` tidak punya `predict_proba`, jadi skor `decision_function` diubah dengan sigmoid standar (sebagai baseline "mentah").
- **Platt**: `CalibratedClassifierCV(method="sigmoid", cv=5)`
- **Isotonic**: `CalibratedClassifierCV(method="isotonic", cv=5)`

TF-IDF dimasukkan ke dalam `Pipeline` sehingga fit hanya pada data latih (tidak ada data leakage).

In [ ]:
class SigmoidSVC(BaseEstimator, ClassifierMixin):
    """LinearSVC + sigmoid(decision_function) sebagai probabilitas 'mentah' (tanpa kalibrasi)."""
    def __init__(self, C=1.0):
        self.C = C
    def fit(self, X, y):
        self.m_ = LinearSVC(C=self.C).fit(X, y)
        self.classes_ = self.m_.classes_
        return self
    def decision_function(self, X):
        return self.m_.decision_function(X)
    def predict_proba(self, X):
        d = self.m_.decision_function(X)
        p = 1.0 / (1.0 + np.exp(-d))
        return np.vstack([1 - p, p]).T
    def predict(self, X):
        return self.m_.predict(X)

def tfidf():
    return TfidfVectorizer(lowercase=True, stop_words="english", ngram_range=(1, 2), min_df=2, sublinear_tf=True)

def base_models(seed):
    return {
        "Naive Bayes":         MultinomialNB(alpha=0.1),
        "Logistic Regression": LogisticRegression(C=10, max_iter=2000, random_state=seed),
        "Linear SVM":          SigmoidSVC(C=1.0),
        "Random Forest":       RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=seed),
    }

def make_pipe(clf):
    return Pipeline([("tfidf", tfidf()), ("clf", clf)])

def make_variant(name, clf, variant, seed):
    if variant == "Raw":
        return make_pipe(clf)
    method = "sigmoid" if variant == "Platt" else "isotonic"
    # Kalibrator membungkus pipeline utuh; LinearSVC dikalibrasi lewat decision_function
    inner = LinearSVC(C=1.0) if name == "Linear SVM" else clf
    return CalibratedClassifierCV(make_pipe(inner), method=method, cv=5)

def ece_score(y_true, y_prob, n_bins=10):
    """Expected Calibration Error, bin lebar sama."""
    bins = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(y_prob, bins[1:-1]), 0, n_bins - 1)
    ece = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.any():
            ece += m.mean() * abs(y_true[m].mean() - y_prob[m].mean())
    return ece

def evaluate(y_true, p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    pred = (p >= 0.5).astype(int)
    return {
        "Accuracy": accuracy_score(y_true, pred),
        "F1": f1_score(y_true, pred),
        "AUC": roc_auc_score(y_true, p),
        "Brier": brier_score_loss(y_true, p),
        "LogLoss": log_loss(y_true, p),
        "ECE": ece_score(y_true, p),
    }

## 3. Jalankan eksperimen (5 seed, split 80/20 stratified)
Pengulangan 5 seed supaya hasil bisa dilaporkan sebagai rata-rata ± simpangan baku (lebih kuat untuk jurnal).

In [ ]:
SEEDS = [0, 1, 2, 3, 4]
VARIANTS = ["Raw", "Platt", "Isotonic"]
rows, store, models0 = [], {}, {}   # store: probabilitas test semua seed (untuk diagram); models0: model seed pertama (studi kasus)

for seed in SEEDS:
    X_tr, X_te, y_tr, y_te = train_test_split(
        df["text"].values, df["y"].values, test_size=0.2, stratify=df["y"].values, random_state=seed)
    for name, clf in base_models(seed).items():
        for var in VARIANTS:
            model = make_variant(name, clf, var, seed)
            model.fit(X_tr, y_tr)
            p = model.predict_proba(X_te)[:, 1]
            res = evaluate(y_te, p)
            res.update({"Seed": seed, "Model": name, "Variant": var})
            rows.append(res)
            s = store.setdefault((name, var), {"y": [], "p": []})
            s["y"].append(y_te.copy()); s["p"].append(p.copy())
            if seed == SEEDS[0]:
                models0[(name, var)] = model
    print(f"seed {seed} selesai")

results = pd.DataFrame(rows)
results.to_csv("outputs/results_all_seeds.csv", index=False)
results.head()

## 4. Tabel ringkasan (mean ± std)

In [ ]:
metrics = ["Accuracy", "F1", "AUC", "Brier", "LogLoss", "ECE"]
agg = results.groupby(["Model", "Variant"])[metrics].agg(["mean", "std"])

order_m = ["Naive Bayes", "Logistic Regression", "Linear SVM", "Random Forest"]
order_v = VARIANTS
summary = pd.DataFrame(index=pd.MultiIndex.from_product([order_m, order_v], names=["Model", "Variant"]))
for m in metrics:
    summary[m] = [f"{agg.loc[(a, b), (m, 'mean')]:.4f} ± {agg.loc[(a, b), (m, 'std')]:.4f}"
                  for a, b in summary.index]
summary.to_csv("outputs/summary_table.csv")
summary

In [ ]:
# Versi numerik (mean) untuk tabel IEEE yang ringkas + file LaTeX
mean_tbl = results.groupby(["Model", "Variant"])[metrics].mean().loc[
    [(a, b) for a in order_m for b in order_v]].round(4)
print(mean_tbl.to_string())
with open("outputs/table_ieee.tex", "w") as f:
    f.write(mean_tbl.to_latex(caption="Performance and calibration metrics (mean over 5 seeds).",
                              label="tab:results", float_format="%.4f"))

## 5. Gaya grafik untuk jurnal
Ukuran gambar dibuat pas untuk ditempel ke template IEEE (lebar penuh = 7.2 inci, satu kolom = 3.5 inci),
sehingga huruf tetap terbaca tanpa perlu diperkecil di Word. Warna aman untuk buta warna (palet Okabe-Ito),
dan tiap varian juga punya bentuk penanda berbeda supaya tetap terbedakan saat dicetak hitam-putih.

In [ ]:
import matplotlib as mpl
mpl.rcParams.update({
    "font.family": "serif", "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 8, "axes.titlesize": 9, "axes.labelsize": 8,
    "legend.fontsize": 7, "xtick.labelsize": 7, "ytick.labelsize": 7,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.5,
})
COLORS  = {"Raw": "#D55E00", "Platt": "#0072B2", "Isotonic": "#009E73"}
MARKERS = {"Raw": "o", "Platt": "s", "Isotonic": "^"}
N_BINS, MIN_N = 10, 5   # 10 bin lebar sama (sama dengan perhitungan ECE); bin dengan < 5 pesan tidak digambar

def pooled(name, var):
    s = store[(name, var)]
    return np.concatenate(s["y"]), np.concatenate(s["p"])

def bin_stats(y, p, n_bins=N_BINS):
    """Per bin lebar sama: (rata-rata probabilitas, fraksi spam, jumlah pesan)."""
    idx = np.clip(np.digitize(p, np.linspace(0, 1, n_bins + 1)[1:-1]), 0, n_bins - 1)
    out = [(p[idx == b].mean(), y[idx == b].mean(), (idx == b).sum()) for b in range(n_bins) if (idx == b).any()]
    return np.array(out)

mean_ece = results.groupby(["Model", "Variant"])["ECE"].mean()   # sama dengan Tabel I

## 6. Reliability diagram (Fig. 1)
- Gabungan prediksi dari kelima split (lebih banyak pesan per bin, lebih stabil daripada satu seed).
- 10 bin lebar sama, konsisten dengan ECE. Luas penanda sebanding dengan jumlah pesan di bin itu; bin dengan < 5 pesan dilewati.
- Panel kecil di bawah tiap diagram menunjukkan jumlah pesan per bin (skala log), sehingga terlihat bahwa hampir semua prediksi berada di dekat 0 atau 1.
- Angka ECE di legenda adalah rata-rata 5 split, sama dengan Tabel I.

In [ ]:
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec

fig = plt.figure(figsize=(7.2, 7.0))
outer = GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.28)
for k, name in enumerate(order_m):
    inner = GridSpecFromSubplotSpec(2, 1, subplot_spec=outer[k], height_ratios=[3.2, 1], hspace=0.08)
    ax, axc = fig.add_subplot(inner[0]), fig.add_subplot(inner[1], sharex=None)
    ax.plot([0, 1], [0, 1], "--", color="0.35", lw=0.8, label="Perfect")
    for var in VARIANTS:
        y, p = pooled(name, var)
        st = bin_stats(y, p)
        keep = st[:, 2] >= MIN_N
        ax.plot(st[keep, 0], st[keep, 1], "-", color=COLORS[var], lw=1.0, alpha=0.9)
        ax.scatter(st[keep, 0], st[keep, 1], s=6 + 90 * np.sqrt(st[keep, 2] / st[:, 2].max()),
                   marker=MARKERS[var], color=COLORS[var], edgecolor="white", linewidth=0.4, zorder=3,
                   label=f"{var} (ECE {mean_ece[(name, var)]:.3f})")
        axc.plot(st[:, 0], st[:, 2], marker=MARKERS[var], ms=3, lw=0.8, color=COLORS[var])
    ax.set_title(name)
    ax.set_xlim(-0.03, 1.03); ax.set_ylim(-0.03, 1.03)
    ax.set_ylabel("Fraction of spam")
    ax.tick_params(labelbottom=False)
    ax.legend(loc="lower right", frameon=False, handletextpad=0.3, borderaxespad=0.2)
    axc.set_yscale("log"); axc.set_xlim(-0.03, 1.03)
    axc.set_xlabel("Mean predicted probability")
    axc.set_ylabel("Count")
fig.savefig("outputs/reliability_diagrams.png", dpi=300, bbox_inches="tight")
fig.savefig("outputs/reliability_diagrams.pdf", bbox_inches="tight")   # vektor, bagus untuk LaTeX
plt.show()

## 7. Histogram distribusi probabilitas (Fig. 2)
Probabilitas model *Raw* dari kelima split digabung. Menunjukkan apakah model menumpuk di 0/1 (percaya diri) atau di tengah (ragu).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(7.2, 4.4), sharex=True)
for ax, name in zip(axes.ravel(), order_m):
    _, p = pooled(name, "Raw")
    ax.hist(p, bins=20, range=(0, 1), color=COLORS["Raw"], edgecolor="white", linewidth=0.4)
    ax.set_title(f"{name} (Raw)")
    ax.set_yscale("log")
for ax in axes[1]: ax.set_xlabel("Predicted P(spam)")
for ax in axes[:, 0]: ax.set_ylabel("Count (log scale)")
fig.tight_layout()
fig.savefig("outputs/prob_histograms.png", dpi=300, bbox_inches="tight")
fig.savefig("outputs/prob_histograms.pdf", bbox_inches="tight")
plt.show()

## 8. Perbandingan ECE antar model (Fig. 3, lebar satu kolom)
Skala log karena ECE SVM mentah (~0.25) jauh di atas yang lain (~0.01). Titik = rata-rata, garis = ± simpangan baku 5 split.

In [ ]:
ece_m = results.pivot_table(index="Model", columns="Variant", values="ECE", aggfunc="mean").loc[order_m, VARIANTS]
ece_s = results.pivot_table(index="Model", columns="Variant", values="ECE", aggfunc="std").loc[order_m, VARIANTS]
fig, ax = plt.subplots(figsize=(3.5, 2.7))
x = np.arange(len(order_m))
for j, var in enumerate(VARIANTS):
    ax.errorbar(x + (j - 1) * 0.18, ece_m[var], yerr=ece_s[var], fmt=MARKERS[var], color=COLORS[var],
                ms=4, capsize=2, lw=0.8, label=var)
ax.set_yscale("log")
from matplotlib.ticker import FuncFormatter, NullFormatter
ax.set_yticks([0.005, 0.01, 0.02, 0.05, 0.1, 0.2]); ax.set_ylim(0.004, 0.35)
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}")); ax.yaxis.set_minor_formatter(NullFormatter())
ax.set_xticks(x); ax.set_xticklabels(["NB", "LR", "SVM", "RF"])
ax.set_ylabel("ECE (log scale, lower is better)")
ax.legend(frameon=False, loc="upper left")
fig.tight_layout()
fig.savefig("outputs/ece_comparison.png", dpi=300, bbox_inches="tight")
fig.savefig("outputs/ece_comparison.pdf", bbox_inches="tight")
plt.show()

## 9. Studi kasus: email "Congratulations! You won $10,000!"
Dipakai untuk menyambungkan eksperimen dengan slide 5 presentasi.

In [ ]:
sample = ["Congratulations! You won $10,000! Click here to claim your prize now",
          "Hey, are we still meeting for lunch tomorrow?"]
case_rows = []
for name in order_m:
    for var in VARIANTS:
        model = models0[(name, var)]
        pr = model.predict_proba(sample)[:, 1]
        case_rows.append({"Model": name, "Variant": var,
                          "P(spam) prize msg": round(pr[0], 3), "P(spam) lunch msg": round(pr[1], 3)})
case_df = pd.DataFrame(case_rows)
case_df.to_csv("outputs/case_study.csv", index=False)
case_df

## 10. Ringkasan otomatis untuk bahan pembahasan

In [ ]:
raw = results[results.Variant == "Raw"].groupby("Model")["ECE"].mean()
best_cal = results.groupby(["Model", "Variant"])["ECE"].mean().unstack()
print("ECE Raw per model:\n", raw.round(4), "\n")
print("Model paling andal (ECE terendah, semua varian):",
      best_cal.stack().idxmin(), "=", round(best_cal.stack().min(), 4))
print("\nPerubahan ECE setelah kalibrasi (negatif = membaik):")
for m in order_m:
    for v in ["Platt", "Isotonic"]:
        print(f"  {m:20s} {v:9s}: {best_cal.loc[m, v] - best_cal.loc[m, 'Raw']:+.4f}")

acc = results.groupby(["Model", "Variant"])["Accuracy"].mean().unstack()
print("\nPerubahan Accuracy setelah kalibrasi:")
for m in order_m:
    for v in ["Platt", "Isotonic"]:
        print(f"  {m:20s} {v:9s}: {acc.loc[m, v] - acc.loc[m, 'Raw']:+.4f}")

In [ ]:
# Unduh semua keluaran (opsional, khusus Colab)
try:
    from google.colab import files
    import shutil
    shutil.make_archive("outputs", "zip", "outputs")
    files.download("outputs.zip")
except Exception:
    print("Bukan di Colab; file ada di folder outputs/")